# Pretrained models

## Setup

In [ ]:
import sys
from pathlib import Path

BRANCH = "colab-training"

if "google.colab" in sys.modules:
    REPO = Path("/content/CNN_for_multilabel_classification")
    if REPO.exists():
        !git -C {REPO} pull -q
    else:
        !git clone -q -b {BRANCH} https://github.com/vumenira/CNN_for_multilabel_classification.git {REPO}
    %pip install -q -r {REPO}/requirements-colab.txt
else:
    REPO = Path.cwd().parent  # running locally from notebooks/

sys.path.insert(0, str(REPO))

from src import colab
colab.setup()

import pandas as pd
import torch
from torchmetrics.functional.classification import multilabel_f1_score

from src.classifier import predict
from src.pretrained import train
from src.config import DATA_DIR, FOLDS_PATH, LABELS, ROOT
from src.postprocess import add_parents, best_thresholds, blend_duplicates, hflip, phash

folds = pd.read_csv(FOLDS_PATH).set_index("image_id")


def val_f1(pred: pd.DataFrame, y: pd.DataFrame) -> float:
    return multilabel_f1_score(torch.tensor(pred.to_numpy(dtype="float32")), torch.tensor(y.to_numpy()), len(LABELS)).item()

## Experiment 1: ConvNeXt-B in22k, 384

In [ ]:
import time

from src import tracking
from src.config import RESULTS_DIR
from src.sync import SyncCheckpoints

SMOKE = True

RUN_NAME = "smoke" if SMOKE else "convnext_b_384_fold0"
SMOKE_KWARGS = dict(max_epochs=2, trainer_kwargs={"limit_train_batches": 30, "limit_val_batches": 10}) if SMOKE else {}
SAVE_DIR = RESULTS_DIR / RUN_NAME
last = SAVE_DIR / "last.ckpt"
resume = last.exists()
if resume:
    colab.restore_sqlite(tracking.MLRUNS_DIR / "mlflow.db", SAVE_DIR / "mlflow.db")
print(f"{'resuming from' if resume else 'new run, saving to'} {SAVE_DIR}")

start = time.time()
trainer = train(
    "convnext_base.fb_in22k_ft_in1k_384",
    size=384,
    cache=False,
    callbacks=[SyncCheckpoints(SAVE_DIR)],
    ckpt_path=last if resume else None,
    run_id=(SAVE_DIR / "run_id.txt").read_text() if resume else None,
    **SMOKE_KWARGS,
)
print(f"{time.time() - start:.0f} s")
if torch.cuda.is_available():
    print(f"max GPU memory {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
dm = trainer.datamodule
RUN_ID = trainer.logger.run_id
best = SAVE_DIR / "best.ckpt"

In [ ]:
val = predict(trainer, dataloaders=dm.val_dataloader(), ckpt_path=best)
test = predict(trainer, datamodule=dm, ckpt_path=best)
val.to_csv(SAVE_DIR / "val_probs.csv")
test.to_csv(SAVE_DIR / "test_probs.csv")
y = folds.loc[val.index, LABELS]
val_f1(val, y)

In [ ]:
path = SAVE_DIR / "submission.csv"
(test >= 0.5).astype(int).rename_axis("image_id").reset_index().to_csv(path, index=False)
if not SMOKE:
    !kaggle competitions submit -c cats-and-dogs -f {path} -m "{RUN_ID} convnext_base in22k 384"

In [ ]:
!kaggle competitions submissions -c cats-and-dogs | head -4

### TTA hflip

In [ ]:
val_tta = (val + predict(trainer, dataloaders=dm.loader(hflip(dm.val_ds)), ckpt_path=best)) / 2
test_tta = (test + predict(trainer, dataloaders=dm.loader(hflip(dm.test_ds)), ckpt_path=best)) / 2
val_f1(val_tta, y)

### pHash ≤ 2

In [ ]:
is_val = (folds.fold == dm.fold).to_numpy()
hashes = phash(DATA_DIR / "train" / f"{i}.jpg" for i in folds.index)
labels = folds[LABELS].to_numpy()

val_dup = blend_duplicates(val_tta, hashes[folds.index.get_indexer(val.index)], hashes[~is_val], labels[~is_val])
test_dup = blend_duplicates(test_tta, phash(DATA_DIR / "test" / f"{i}.jpg" for i in test.index), hashes, labels)
val_f1(val_dup, y)

### Per-class thresholds

In [ ]:
thresholds = best_thresholds(val_dup, y)
print(thresholds.round(2).to_dict())
val_f1(val_dup >= thresholds, y)

### Hierarchy

In [ ]:
val_f1(add_parents(val_dup >= thresholds), y)

### Submit

In [ ]:
path = SAVE_DIR / "submission_post.csv"
add_parents(test_dup >= thresholds).astype(int).rename_axis("image_id").reset_index().to_csv(path, index=False)
if not SMOKE:
    !kaggle competitions submit -c cats-and-dogs -f {path} -m "{RUN_ID} convnext_base in22k 384 + post"

In [ ]:
!kaggle competitions submissions -c cats-and-dogs | head -4

## Finish: wait until Drive has every file

In [ ]:
if colab.in_colab():
    from google.colab import drive
    drive.flush_and_unmount()